# 🚀 Tank Battle 2D — High-Performance Cloud Training (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BohatyrovIvanUcu/tank_battle_playground/blob/main/colab_trainer.ipynb)

Цей блокнот дозволяє запустити паралельне тренування генетичного алгоритму або нейромережі на потужностях **Google Colab (NVIDIA A100 / High-RAM CPU з 16+ ядрами)**.

### ⚡ Особливості:
- **Багатопроцесорна паралелізація (`--workers 16`)**: популяція з 25 танків оцінюється паралельно на всіх ядрах, що прискорює навчання в **10–15 разів** (1000 поколінь пролітають за 2–3 хвилини).
- **Self-Play (коеволюція)** або навчання проти `rule_based` бота.
- **Вбудований 2D рендерер прямо в блокноті**: запис поєдинку в анімований GIF / HTML5 без потреби в графічному сервері (Headless).
- **Завантаження готового чемпіона в 1 клік** для змагання проти друга.

## 1. 🖥️ Перевірка заліза (GPU & CPU)

In [ ]:
# Перевірка відеокарти (якщо обрано GPU Runtime) та кількості ядер CPU
!nvidia-smi
!lscpu | grep -E "Model name|CPU\(s\):|Thread|Core"

import multiprocessing as mp
print(f"\n[+] Доступно логічних ядер процесора для паралельного тренування: {mp.cpu_count()}")

## 2. 🦀 Встановлення Rust та Maturin
Рушій фізики та симуляції написаний на Rust, тому ми встановлюємо офіційний компілятор `rustc` і збирач `maturin` (~20-30 секунд).

In [ ]:
# Встановлення Rustup та cargo
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] = f"/root/.cargo/bin:{os.environ['PATH']}"
!pip install --quiet maturin numpy matplotlib pillow imageio
!rustc --version

## 3. 📦 Клонування репозиторію та збірка рушія

In [ ]:
# Клонуємо найсвіжішу версію репозиторію
!rm -rf tank_battle_playground
!git clone https://github.com/BohatyrovIvanUcu/tank_battle_playground.git
%cd tank_battle_playground

# Збираємо нативний біндінґ для Python у режимі максимальної оптимізації
!maturin develop --release

## 4. ⚔️ Запуск паралельного тренування (Self-Play або Боти)

Тут ви можете налаштувати параметри тренування:
- `GENERATIONS`: кількість поколінь (наприклад, 1000 чи навіть 3000);
- `POPULATION`: кількість особин у популяції (25 або 50);
- `OPPONENT`: `self` (коеволюція проти пулу минулих чемпіонів) або `rule_based`;
- `WORKERS`: кількість ядер (ставимо всі доступні ядра для ультра-швидкості).

In [ ]:
import multiprocessing as mp

GENERATIONS = 1000         # Скільки поколінь тренувати
POPULATION = 25            # Розмір популяції
OPPONENT = "self"          # "self" (грати проти себе) або "rule_based"
WORKERS = mp.cpu_count()   # Використовувати всі ядра CPU паралельно
CHECKPOINT_INTERVAL = 100  # Зберігати модель кожні N поколінь

# Якщо хочете продовжити з існуючого чекпоінту, додайте --resume checkpoints/best_model.npz
cmd = f"python3 python/train_ga.py --adapter combat --opponent {OPPONENT} --generations {GENERATIONS} --population {POPULATION} --workers {WORKERS} --checkpoint-interval {CHECKPOINT_INTERVAL}"

print(f"[+] Запуск команди: {cmd}\n")
!{cmd}

## 5. 🎬 Візуалізація бою прямо в блокноті (Headless Renderer)
Оскільки в Colab немає фізичного монітора для виведення вікна `minifb`, цей код автоматично проводить поєдинок, рендерить кадри з телеметрії та відображає анімований бій прямо тут!

In [ ]:
import math
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import Image as IPyImage, display
from tank_env import TankEnvironment, load_agent

def render_match_gif(agent_a_path, agent_b_path, seed=42, max_steps=400, output_gif="match.gif"):
    env = TankEnvironment(max_steps=max_steps)
    a1 = load_agent(agent_a_path)
    a2 = load_agent(agent_b_path)
    obs1, obs2 = env.reset(seed=seed)

    width, height = 800, 533  # Масштабований розмір для Colab
    scale_x, scale_y = width / 1200.0, height / 800.0

    frames = []
    for step in range(max_steps):
        act1 = a1.act(obs1)
        act2 = a2.act(obs2)
        obs1, obs2, r, done, info = env.step(act1, act2)

        # Рендеримо кожен 2-й кадр для плавної анімації
        if step % 2 == 0:
            img = Image.new("RGB", (width, height), "#141923")
            draw = ImageDraw.Draw(img)

            # Малюємо перешкоди
            for o in obs1.get("obstacles", []):
                ox, oy, ow, oh = o[0] * 1200 * scale_x, o[1] * 800 * scale_y, o[2] * 1200 * scale_x, o[3] * 800 * scale_y
                draw.rectangle([ox, oy, ox + ow, oy + oh], fill="#2d3748", outline="#4a5568", width=2)

            # Малюємо Танк 1 (Синій/Зелений)
            t1_x, t1_y = obs1["x"] * 1200 * scale_x, obs1["y"] * 800 * scale_y
            r1 = 18.0 * scale_x
            draw.ellipse([t1_x - r1, t1_y - r1, t1_x + r1, t1_y + r1], fill="#3182ce", outline="#63b3ed", width=2)
            # Дуло 1
            turr1 = obs1["turret_angle"]
            draw.line([t1_x, t1_y, t1_x + math.cos(turr1) * 26 * scale_x, t1_y + math.sin(turr1) * 26 * scale_y], fill="#63b3ed", width=3)

            # Малюємо Танк 2 (Червоний/Помаранчевий)
            t2_x, t2_y = obs2["x"] * 1200 * scale_x, obs2["y"] * 800 * scale_y
            draw.ellipse([t2_x - r1, t2_y - r1, t2_x + r1, t2_y + r1], fill="#e53e3e", outline="#feb2b2", width=2)
            # Дуло 2
            turr2 = obs2["turret_angle"]
            draw.line([t2_x, t2_y, t2_x + math.cos(turr2) * 26 * scale_x, t2_y + math.sin(turr2) * 26 * scale_y], fill="#feb2b2", width=3)

            # Кулі
            for b in obs1.get("bullets", []):
                bx = (obs1["x"] + b[0]) * 1200 * scale_x
                by = (obs1["y"] + b[1]) * 800 * scale_y
                draw.ellipse([bx - 3, by - 3, bx + 3, by + 3], fill="#ecc94b")

            # HP бари зверху
            draw.rectangle([20, 20, 20 + obs1['hp'] * 2, 32], fill="#3182ce")
            draw.rectangle([width - 220, 20, width - 220 + obs2['hp'] * 2, 32], fill="#e53e3e")

            frames.append(img)

        if done:
            break

    frames[0].save(output_gif, save_all=True, append_images=frames[1:], duration=32, loop=0)
    print(f"[+] Поєдинок збережено в {output_gif}! Кроків: {step}, HP1: {obs1['hp']}, HP2: {obs2['hp']}")
    return output_gif

# Рендеримо бій нашого найкращого агента проти Rule-Based бота
gif_path = render_match_gif("checkpoints/agent.py", "rule_based", seed=99)
display(IPyImage(gif_path))

## 6. 📥 Завантаження натренованого чемпіона
Ця клітинка архівує ваги найкращої моделі (`best_model.npz`) та код агента (`agent.py`) і завантажує `.zip` прямо на ваш комп'ютер.

In [ ]:
!zip -r tank_champion.zip checkpoints/best_model.npz checkpoints/agent.py

from google.colab import files
files.download("tank_champion.zip")
print("[✓] Файл tank_champion.zip готовий! Просто розпакуйте його вміст у теку checkpoints/ на вашому ПК.")